In [2]:
!pip install -qU langchain langchain-community langchain-google-genai pypdf

In [3]:
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("Gemini API key loaded!")

Gemini API key loaded!


In [4]:
from google.colab import files

print(" Please upload a .txt, .pdf, or .csv file:")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print(f"\n File uploaded: {file_name}")

 Please upload a .txt, .pdf, or .csv file:


Saving DBMS TE_SYLLABUS.txt to DBMS TE_SYLLABUS.txt

 File uploaded: DBMS TE_SYLLABUS.txt


In [6]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, CSVLoader

if file_name.endswith(".txt"):
    loader = TextLoader(file_name, encoding="utf-8")
    file_type = "TXT"
elif file_name.endswith(".pdf"):
    loader = PyPDFLoader(file_name)
    file_type = "PDF"
elif file_name.endswith(".csv"):
    loader = CSVLoader(file_name)
    file_type = "CSV"
else:
    raise ValueError(f" Unsupported file type: {file_name}. Please upload a .txt, .pdf, or .csv file.")

docs = loader.load()

print(f" File Type     : {file_type}")
print(f" type(docs)    : {type(docs)}")
print(f" len(docs)     : {len(docs)}")
print(f"\n Preview of docs[0].page_content (first 500 chars):")
print("-" * 50)
print(docs[0].page_content[:500])

 File Type     : TXT
 type(docs)    : <class 'list'>
 len(docs)     : 1

 Preview of docs[0].page_content (first 500 chars):
--------------------------------------------------
Curriculum for Third Year of Information Technology (2019 Course), Savitribai Phule Pune University 
TE (Information Technology) Syllabus (2019 Course) 24 
 
 
 
Savitribai Phule Pune University, Pune 
Third Year Information Technology (2019 Course) 
314445(B): Elective -I : Advanced Database Management System 
Teaching Scheme: Credit Scheme: Examination Scheme: 
Theory (TH) : 3 hrs/week 03 Credits Mid_Semester : 30 Marks 
End_Semester : 70 Marks 
Prerequisite Courses: 
1. Database Management Sy


In [7]:

combined_content = "\n\n".join([doc.page_content for doc in docs])

print(f"Total content length: {len(combined_content)} characters")
print(f"Combined from {len(docs)} document chunk(s)")

Total content length: 2491 characters
Combined from 1 document chunk(s)


In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=GEMINI_API_KEY
)

prompt_template = PromptTemplate(
    input_variables=["content"],
    template="""
You are an expert document analyst. Read the content below carefully and generate a clear, concise summary.

Content:
{content}

Instructions:
- Write exactly 10 bullet points
- Each bullet point should capture a distinct key insight or fact
- Be specific, not vague
- Keep each point to 1–2 sentences
- Do not repeat information across points

10-Point Bullet Summary:
"""
)

chain = prompt_template | llm | StrOutputParser()

print(" Summarization chain ready!")

 Summarization chain ready!


In [11]:
from IPython.display import Markdown, display

print(f" Summarizing your {file_type} file...\n")

content_to_summarize = combined_content[:12000]

summary = chain.invoke({"content": content_to_summarize})

print("=" * 60)
print(f" Summary of: {file_name}")
print("=" * 60)
display(Markdown(summary))

 Summarizing your TXT file...

 Summary of: DBMS TE_SYLLABUS.txt


Here is a 10-point bullet summary of the provided content:

*   The document outlines the syllabus for the Third Year Information Technology (2019 Course) elective, Advanced Database Management System (314445(B)), at Savitribai Phule Pune University.
*   This elective course is assigned 03 credits and requires 3 hours of theory instruction per week.
*   The examination scheme for the course includes a 30-mark Mid-Semester assessment and a 70-mark End-Semester assessment.
*   A prerequisite for enrolling in this course is a foundational understanding of Database Management Systems.
*   One key objective is to understand the fundamental concepts of both Relational and Object-oriented databases.
*   The course aims for students to learn and comprehend various Parallel and Distributed Database Architectures and their applications.
*   Students are expected to understand and apply the basic concepts, categories, and tools of NoSQL Databases.
*   Another objective is to learn and understand Data warehouse and OLAP Architectures and their respective applications.
*   Unit I, "Review of Relational Data Model and Relational Database Constraints" (06 hrs), covers relational model concepts, update operations, and an overview of object-oriented concepts like encapsulation and polymorphism.
*   Unit II, "Parallel and Distributed Databases" (06 hrs), introduces parallel database architectures, query evaluation, distributed DBMS architectures, query processing, and distributed concurrency control.